# 01 · Entendimiento de los datos, EDA y limpieza

**Fase CRISP-DM:** entendimiento y preparación de los datos.

Se trabaja con el dataset **CICIoMT2024** (tráfico Wi-Fi y MQTT de
dispositivos médicos) y tres tipos de tráfico:

| `clase` | Tráfico | `etiqueta` (binaria) |
|---|---|---|
| 0 | Benigno | 0 |
| 1 | DDoS SYN | 1 |
| 2 | DDoS UDP | 1 |

Los CSV deben estar en `data/raw/CICIoMT2024/train/` y `.../test/`
(ver `DESCARGA_DATOS.txt`). Este notebook genera
`data/processed/train_limpio.parquet` y `data/processed/test_limpio.parquet`.

In [ ]:
import sys
from pathlib import Path

# Permite importar src/ al ejecutar desde notebooks/ o desde la raíz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import (
    CLASES, verificar_archivos, cargar_particion, columnas_caracteristicas,
    resumen_calidad, limpiar, guardar_procesado,
)

pd.set_option("display.max_columns", 60)
display(verificar_archivos())

## 1. Estructura de los archivos

In [ ]:
train = cargar_particion("train")
test = cargar_particion("test")
variables = columnas_caracteristicas(train)

print("Entrenamiento — filas y columnas:", train.shape)
print("Prueba oficial — filas y columnas:", test.shape)
print("Mismas columnas y orden en train y test:",
      list(train.columns) == list(test.columns))

print(f"\n{len(variables)} variables de flujo:")
print(variables)

print("\nTipos de datos:")
print(train[variables].dtypes.value_counts())

In [ ]:
display(train.groupby("clase").head(2).set_index("clase").rename(index=CLASES))

El dataset no incluye direcciones IP, puertos ni marcas de tiempo:
todas las columnas son estadísticas de flujo (tamaños, tasas, banderas
TCP e indicadores de protocolo), como propone el informe.

## 2. Distribución de clases

In [ ]:
conteo = pd.DataFrame({
    "Entrenamiento": train["clase"].value_counts(),
    "Prueba oficial": test["clase"].value_counts(),
}).sort_index().rename(index=CLASES)
proporcion = conteo / conteo.sum()

display(conteo.join(proporcion.round(3).add_suffix(" (proporción)")))

proporcion.T.plot.bar(
    stacked=True, figsize=(7, 4), rot=0,
    title="Proporción de clases por partición"
)
plt.ylabel("Proporción de registros")
plt.legend(title="Clase", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

Las proporciones de clase no son iguales en entrenamiento y en la prueba
oficial. En la prueba predomina el tráfico de ataque, por lo que la
accuracy y la precisión de DDoS pueden verse altas incluso con un
modelo trivial. Por eso se reportan también F1 macro, recall por clase
y tasa de falsos positivos.

## 3. Calidad de los datos

In [ ]:
calidad = pd.DataFrame({
    "Entrenamiento": resumen_calidad(train),
    "Prueba oficial": resumen_calidad(test),
})
display(calidad)

faltantes = train[variables].isna().sum()
print("Columnas con valores faltantes en entrenamiento:")
print(faltantes[faltantes > 0])

print("\nDuplicados por clase en entrenamiento:")
print(train.loc[train.duplicated(), "clase"].map(CLASES).value_counts())

## 4. Variables constantes y redundantes

In [ ]:
resumen = pd.DataFrame({
    "Valores distintos": train[variables].nunique(),
    "Valor más frecuente (proporción)": train[variables].apply(
        lambda s: s.value_counts(normalize=True).iloc[0]
    ),
}).sort_values("Valores distintos")
display(resumen.head(15).round(4))

identicas = [
    (a, b)
    for i, a in enumerate(variables)
    for b in variables[i + 1:]
    if train[a].equals(train[b])
]
print("Pares de columnas idénticas:", identicas)

Las columnas constantes no aportan información y las idénticas son
redundantes. Se eliminan en el notebook 02, decidiendo solo con el
conjunto de entrenamiento.

## 5. Estadísticas por clase

In [ ]:
destacadas = [
    "Header_Length", "Protocol Type", "Duration", "Rate",
    "syn_flag_number", "ack_flag_number", "rst_flag_number",
    "TCP", "UDP", "Tot size", "AVG", "Std", "IAT",
]
display(
    train.groupby("clase")[destacadas].median()
    .rename(index=CLASES).T.round(3)
    .rename_axis("Mediana por clase")
)

In [ ]:
indicadores = (
    [c for c in variables if c.endswith("_flag_number")]
    + ["HTTP", "HTTPS", "DNS", "TCP", "UDP", "ICMP", "ARP", "IPv", "LLC"]
)
medias = train.groupby("clase")[indicadores].mean().rename(index=CLASES)

fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(medias, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(indicadores)), indicadores, rotation=60, ha="right")
ax.set_yticks(range(len(medias)), medias.index)
for i in range(medias.shape[0]):
    for j in range(medias.shape[1]):
        valor = medias.iat[i, j]
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=7,
                color="white" if valor > 0.6 else "black")
fig.colorbar(im, ax=ax, label="Proporción media")
ax.set_title("Banderas TCP y protocolos por clase")
plt.tight_layout()
plt.show()

Este gráfico muestra qué banderas y protocolos caracterizan a cada
ataque. Se espera que DDoS SYN tenga una alta proporción de la bandera
SYN y DDoS UDP una alta proporción de UDP. Si una sola variable separa
casi perfectamente las clases, el modelo puede depender de ese "atajo",
lo que es relevante al evaluar la generalización entre dominios.

## 6. Distribuciones por clase

Se usa una muestra de hasta 50.000 registros por clase y la escala
`log(1 + x)`, porque variables como `Rate` tienen colas muy largas.

In [ ]:
muestra = pd.concat([
    g.sample(min(len(g), 50_000), random_state=42)
    for _, g in train.groupby("clase")
]).replace([np.inf, -np.inf], np.nan)

columnas_grafico = ["Header_Length", "Rate", "Tot size", "Std", "Duration", "IAT"]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, columna in zip(axes.ravel(), columnas_grafico):
    for clase, nombre in CLASES.items():
        valores = muestra.loc[muestra["clase"] == clase, columna].dropna()
        ax.hist(np.log1p(valores.clip(lower=0)), bins=60, alpha=0.5,
                density=True, label=nombre)
    ax.set_title(columna)
    ax.set_xlabel("log(1 + valor)")
axes[0, 0].legend()
fig.suptitle("Distribución de variables seleccionadas por clase")
plt.tight_layout()
plt.show()

## 7. Correlación entre variables

In [ ]:
no_constantes = [c for c in variables if muestra[c].nunique() > 1]
correlacion = muestra[no_constantes].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(correlacion, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(no_constantes)), no_constantes, rotation=90, fontsize=7)
ax.set_yticks(range(len(no_constantes)), no_constantes, fontsize=7)
fig.colorbar(im, ax=ax, label="Correlación de Pearson")
ax.set_title("Matriz de correlación (muestra de entrenamiento)")
plt.tight_layout()
plt.show()

pares = correlacion.where(
    np.triu(np.ones(correlacion.shape, dtype=bool), k=1)
).stack()
display(
    pares[pares.abs() > 0.95]
    .sort_values(key=np.abs, ascending=False)
    .rename("Correlación").to_frame().round(4)
)

## 8. Limpieza y guardado

- **Entrenamiento:** infinitos → NaN, se eliminan filas con
  características idénticas y clases distintas, y duplicados exactos.
  Así una misma fila no puede quedar en entrenamiento y validación.
- **Prueba oficial:** solo infinitos → NaN. No se eliminan filas, para
  evaluar el tráfico tal como se registró.

Los valores NaN se imputan con la mediana dentro de los pipelines de
modelado, ajustando la imputación solo con entrenamiento.

In [ ]:
train_limpio, informe_train = limpiar(train, eliminar_filas=True)
test_limpio, informe_test = limpiar(test, eliminar_filas=False)

display(pd.DataFrame({
    "Entrenamiento": informe_train,
    "Prueba oficial": informe_test,
}))

print("Registros de entrenamiento por clase tras la limpieza:")
print(train_limpio["clase"].map(CLASES).value_counts())

print("\nGuardado:", guardar_procesado(train_limpio, "train_limpio"))
print("Guardado:", guardar_procesado(test_limpio, "test_limpio"))

## Conclusiones del EDA

Completar con los resultados obtenidos al ejecutar el notebook:

1. **Calidad:** cantidad de faltantes, infinitos, duplicados y filas
   contradictorias, y cuántos registros se eliminaron.
2. **Desbalance:** diferencia de proporciones entre entrenamiento y
   prueba oficial, y por qué se usan F1 macro y tasa de falsos positivos.
3. **Separabilidad:** variables que distinguen claramente cada ataque
   (banderas SYN, protocolo UDP, tasas) y el riesgo de "atajos".
4. **Redundancia:** columnas constantes, idénticas o muy correlacionadas
   que se tratarán en el notebook 02.